In [1]:
import sys

print("Python executable:")
print(sys.executable)

print("\nPython version:")
print(sys.version)

Python executable:
c:\Users\KETAKI PATIL\NLP_DL-project\AI-Legal-Document-Analyzer\.venv\Scripts\python.exe

Python version:
3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]


In [2]:
from pathlib import Path
import re
import json
import sys

import pymupdf
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print("Python:", sys.version)
print("Environment:", sys.executable)
print("Baseline environment ready.")

Python: 3.10.11 (tags/v3.10.11:7d4cc5a, Apr  5 2023, 00:38:17) [MSC v.1929 64 bit (AMD64)]
Environment: c:\Users\KETAKI PATIL\NLP_DL-project\AI-Legal-Document-Analyzer\.venv\Scripts\python.exe
Baseline environment ready.


In [3]:
PROJECT_ROOT = Path.cwd().parent

PDF_PATH = PROJECT_ROOT / "data" / "raw" / "sample-independent-contractor-agreement.pdf"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("PDF:", PDF_PATH)
print("PDF exists:", PDF_PATH.exists())

Project root: c:\Users\KETAKI PATIL\NLP_DL-project\AI-Legal-Document-Analyzer
PDF: c:\Users\KETAKI PATIL\NLP_DL-project\AI-Legal-Document-Analyzer\data\raw\sample-independent-contractor-agreement.pdf
PDF exists: True


In [4]:
doc = pymupdf.open(PDF_PATH)

pages = []

for page_number, page in enumerate(doc, start=1):
    text = page.get_text("text")
    
    pages.append({
        "page": page_number,
        "text": text.strip()
    })

doc.close()

pages_df = pd.DataFrame(pages)

print("Number of pages:", len(pages_df))
print("\nCharacters per page:")
print(pages_df["text"].str.len().to_string(index=False))

Number of pages: 3

Characters per page:
1590
1603
1426


In [5]:
for _, row in pages_df.iterrows():
    print("=" * 80)
    print(f"PAGE {row['page']}")
    print("=" * 80)
    print(row["text"][:2500])
    print()

PAGE 1
Sample Independent Contractor 
Agreement 
 
Disclaimer: What follows is a general template designed to give business owners a sense of what is 
typically included in an independent contractor agreement. None of the information that follows 
qualifies as legal advice. Consult legal counsel before using any sample agreement. 
  
Sample Independent Contractor Agreement 
 
This Independent Contractor Agreement ("Agreement") is entered into as of [Insert Date], by and between: 
 
[Company Name], a [State/Country] entity with a principal place of business at [Company Address], 
 
and 
 
[Contractor Name], an independent contractor residing at [Contractor Address]. 
1. Scope of Work 
The Contractor agrees to provide the following services: 
 
[Describe the work to be performed, e.g., "Website design, development, and SEO optimization."] 
 
The Contractor will deliver all services provided in writing in accordance with any project milestones, 
deadlines, or specifications provided by th

In [6]:
#Cell 5 — Basic text statistics
full_text = "\n".join(pages_df["text"])

words = re.findall(r"\b\w+\b", full_text)

print("Pages:", len(pages_df))
print("Characters:", len(full_text))
print("Words:", len(words))
print("Approx. lines:", len(full_text.splitlines()))

Pages: 3
Characters: 4621
Words: 643
Approx. lines: 123


In [7]:
#Clean the extracted text
def clean_text(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

cleaned_pages = []

for _, row in pages_df.iterrows():
    cleaned_pages.append({
        "page": row["page"],
        "text": clean_text(row["text"])
    })

cleaned_df = pd.DataFrame(cleaned_pages)

cleaned_full_text = "\n\n".join(cleaned_df["text"])

print(cleaned_full_text[:3000])

Sample Independent Contractor 
Agreement 
 
Disclaimer: What follows is a general template designed to give business owners a sense of what is 
typically included in an independent contractor agreement. None of the information that follows 
qualifies as legal advice. Consult legal counsel before using any sample agreement. 
 
Sample Independent Contractor Agreement 
 
This Independent Contractor Agreement ("Agreement") is entered into as of [Insert Date], by and between: 
 
[Company Name], a [State/Country] entity with a principal place of business at [Company Address], 
 
and 
 
[Contractor Name], an independent contractor residing at [Contractor Address]. 
1. Scope of Work 
The Contractor agrees to provide the following services: 
 
[Describe the work to be performed, e.g., "Website design, development, and SEO optimization."] 
 
The Contractor will deliver all services provided in writing in accordance with any project milestones, 
deadlines, or specifications provided by the Client

In [8]:
#Basic clause/section segmentation
pattern = r"(?m)^\s*(\d+)\.\s+([A-Z][^\n]+)"

matches = list(re.finditer(pattern, cleaned_full_text))

clauses = []

for i, match in enumerate(matches):
    start = match.start()
    end = matches[i + 1].start() if i + 1 < len(matches) else len(cleaned_full_text)

    section_number = match.group(1)
    section_title = match.group(2).strip()

    section_text = cleaned_full_text[start:end].strip()

    clauses.append({
        "clause_id": int(section_number),
        "title": section_title,
        "text": section_text
    })

clauses_df = pd.DataFrame(clauses)

print("Number of detected sections:", len(clauses_df))
display(clauses_df[["clause_id", "title"]])

Number of detected sections: 9


,clause_id,title
0,1,Scope of Work
1,2,Term and Termination
2,3,Payment Terms
3,4,Independent Contractor Status
4,5,Intellectual Property
5,6,Confidentiality
6,7,Compliance with Laws
7,8,Indemnification
8,9,Miscellaneous


In [9]:
#Cell 8 — Inspect clauses
for _, row in clauses_df.iterrows():
    print("=" * 80)
    print(f"CLAUSE {row['clause_id']}: {row['title']}")
    print("=" * 80)
    print(row["text"])
    print()

CLAUSE 1: Scope of Work
1. Scope of Work 
The Contractor agrees to provide the following services: 
 
[Describe the work to be performed, e.g., "Website design, development, and SEO optimization."] 
 
The Contractor will deliver all services provided in writing in accordance with any project milestones, 
deadlines, or specifications provided by the Client.

CLAUSE 2: Term and Termination
2. Term and Termination 
This Agreement will begin on [Start Date] and continue until the Services are completed unless terminated 
earlier by either party with [7/14] days’ written notice. 
 
The Client may terminate immediately for breach, nonperformance, or ethical concerns. Upon termination, 
the Contractor will be paid for Services rendered through the date of termination.

CLAUSE 3: Payment Terms
3. Payment Terms 
The Client agrees to pay the Contractor: 
 
●​ Rate: $[Rate] per [hour/project/milestone] 
 
 
This document is for informational purposes only. Seek professional advice before using it

In [10]:
#Cell 9 — Generate semantic embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

clause_embeddings = embedding_model.encode(
    clauses_df["text"].tolist(),
    convert_to_numpy=True,
    show_progress_bar=True
)

print("Embedding shape:", clause_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\KETAKI PATIL\NLP_DL-project\AI-Legal-Document-Analyzer\.venv\lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\KETAKI PATIL\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (9, 384)


In [11]:
#Cell 10 — Semantic clause search
def search_clauses(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    )

    similarities = cosine_similarity(
        query_embedding,
        clause_embeddings
    )[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = clauses_df.iloc[top_indices].copy()
    results["similarity"] = similarities[top_indices]

    return results[["clause_id", "title", "similarity", "text"]]

In [12]:
query = "What happens if the contractor wants to end the agreement?"

results = search_clauses(query)

display(results)

,clause_id,title,similarity,text
1,2,Term and Termination,0.711086,2. Term and Termination \nThis Agreement will ...
7,8,Indemnification,0.559510,8. Indemnification \nThe Contractor shall inde...
8,9,Miscellaneous,0.547226,9. Miscellaneous \n●​ Entire Agreement: This d...


In [13]:
display(search_clauses("Who owns the work created by the contractor?"))

,clause_id,title,similarity,text
3,4,Independent Contractor Status,0.561604,4. Independent Contractor Status \nThe Contrac...
4,5,Intellectual Property,0.522231,"5. Intellectual Property \nAll work product, m..."
0,1,Scope of Work,0.481422,1. Scope of Work \nThe Contractor agrees to pr...


In [14]:
display(search_clauses("What information must the contractor keep secret?"))

,clause_id,title,similarity,text
5,6,Confidentiality,0.758276,6. Confidentiality \nThe Contractor agrees not...
6,7,Compliance with Laws,0.533117,7. Compliance with Laws \nThe Contractor agree...
3,4,Independent Contractor Status,0.503975,4. Independent Contractor Status \nThe Contrac...


In [15]:
#Cell 11 — Baseline risk keyword analysis
RISK_KEYWORDS = {
    "HIGH": [
        "indemnify",
        "hold harmless",
        "sole and exclusive property",
        "no rights",
        "terminate immediately",
        "liabilities"
    ],
    "MEDIUM": [
        "termination",
        "confidential",
        "tax obligations",
        "comply",
        "dispute",
        "arbitration",
        "governing law"
    ]
}

def assess_risk(text):
    text_lower = text.lower()

    high_matches = [
        keyword for keyword in RISK_KEYWORDS["HIGH"]
        if keyword in text_lower
    ]

    medium_matches = [
        keyword for keyword in RISK_KEYWORDS["MEDIUM"]
        if keyword in text_lower
    ]

    if high_matches:
        level = "HIGH"
    elif medium_matches:
        level = "MEDIUM"
    else:
        level = "LOW"

    return level, high_matches, medium_matches

In [16]:
risk_results = []

for _, row in clauses_df.iterrows():
    level, high_matches, medium_matches = assess_risk(row["text"])

    risk_results.append({
        "clause_id": row["clause_id"],
        "title": row["title"],
        "risk_level": level,
        "high_risk_indicators": ", ".join(high_matches),
        "medium_risk_indicators": ", ".join(medium_matches)
    })

risk_df = pd.DataFrame(risk_results)

display(risk_df)

,clause_id,title,risk_level,high_risk_indicators,medium_risk_indicators
0,1,Scope of Work,LOW,,
1,2,Term and Termination,HIGH,terminate immediately,termination
2,3,Payment Terms,LOW,,
3,4,Independent Contractor Status,LOW,,
4,5,Intellectual Property,HIGH,no rights,
5,6,Confidentiality,MEDIUM,,"termination, confidential"
6,7,Compliance with Laws,MEDIUM,,comply
7,8,Indemnification,HIGH,"indemnify, hold harmless, liabilities",
8,9,Miscellaneous,MEDIUM,,"dispute, arbitration, governing law"


In [17]:
#Cell 12 — Identify important clauses
important_clauses = risk_df[
    risk_df["risk_level"].isin(["HIGH", "MEDIUM"])
].copy()

print("Important/risk-relevant clauses:")
display(important_clauses)

Important/risk-relevant clauses:


,clause_id,title,risk_level,high_risk_indicators,medium_risk_indicators
1,2,Term and Termination,HIGH,terminate immediately,termination
4,5,Intellectual Property,HIGH,no rights,
5,6,Confidentiality,MEDIUM,,"termination, confidential"
6,7,Compliance with Laws,MEDIUM,,comply
7,8,Indemnification,HIGH,"indemnify, hold harmless, liabilities",
8,9,Miscellaneous,MEDIUM,,"dispute, arbitration, governing law"


In [18]:
#Cell 13 — Create baseline report data
baseline_summary = {
    "document": PDF_PATH.name,
    "pages": len(pages_df),
    "characters": len(cleaned_full_text),
    "words": len(re.findall(r"\b\w+\b", cleaned_full_text)),
    "detected_sections": len(clauses_df),
    "embedding_model": "all-MiniLM-L6-v2",
    "risk_method": "keyword-based baseline",
    "classification_model": "None - rule-based baseline"
}

print(json.dumps(baseline_summary, indent=4))

{
    "document": "sample-independent-contractor-agreement.pdf",
    "pages": 3,
    "characters": 4621,
    "words": 643,
    "detected_sections": 9,
    "embedding_model": "all-MiniLM-L6-v2",
    "risk_method": "keyword-based baseline",
    "classification_model": "None - rule-based baseline"
}


In [19]:
#Cell 14 — Save baseline outputs
clauses_df.to_csv(
    PROCESSED_DIR / "baseline_clauses.csv",
    index=False
)

risk_df.to_csv(
    PROCESSED_DIR / "baseline_risk_analysis.csv",
    index=False
)

with open(PROCESSED_DIR / "baseline_summary.json", "w") as f:
    json.dump(baseline_summary, f, indent=4)

print("Baseline outputs saved successfully.")

Baseline outputs saved successfully.


In [20]:
#Cell 15 — Final baseline demonstration
print("=" * 70)
print("AI LEGAL DOCUMENT ANALYZER — BASELINE")
print("=" * 70)

print(f"Document: {PDF_PATH.name}")
print(f"Pages: {baseline_summary['pages']}")
print(f"Words: {baseline_summary['words']}")
print(f"Sections detected: {baseline_summary['detected_sections']}")

print("\nRisk-relevant clauses:")
for _, row in risk_df.iterrows():
    if row["risk_level"] != "LOW":
        print(f"- {row['title']}: {row['risk_level']}")

print("\nSemantic search test:")
test_query = "Who owns the work produced by the contractor?"
result = search_clauses(test_query, top_k=1)

print("Query:", test_query)
print("Retrieved clause:", result.iloc[0]["title"])
print("Similarity:", round(result.iloc[0]["similarity"], 4))

print("\nBASELINE COMPLETE")

AI LEGAL DOCUMENT ANALYZER — BASELINE
Document: sample-independent-contractor-agreement.pdf
Pages: 3
Words: 643
Sections detected: 9

Risk-relevant clauses:
- Term and Termination: HIGH
- Intellectual Property: HIGH
- Confidentiality: MEDIUM
- Compliance with Laws: MEDIUM
- Indemnification: HIGH
- Miscellaneous: MEDIUM

Semantic search test:
Query: Who owns the work produced by the contractor?
Retrieved clause: Independent Contractor Status
Similarity: 0.5407

BASELINE COMPLETE
